# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")
print("DuckDB ready, HF secret registered.")

MONTH = "2026-03"
BASE = "hf://datasets/FlyRank/internship-warehouse"
print(f"Working month: {MONTH}")

DuckDB ready, HF secret registered.
Working month: 2026-03


In [5]:
test = con.sql(f"SELECT COUNT(*) AS n FROM read_parquet('{BASE}/fact_content_daily_performance_sample.parquet')").df()
test

,n
0,11694072


In [6]:
# Confirm what columns dim_content actually has before relying on them below
dim_content_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{BASE}/dim_content.parquet')").df()
dim_content_cols

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,keyword_hash_id,VARCHAR,YES,None,None,None
3,url_hash_id,VARCHAR,YES,None,None,None
4,keyword_char_count,BIGINT,YES,None,None,None
5,keyword_token_count,BIGINT,YES,None,None,None
6,url_char_count,BIGINT,YES,None,None,None
7,content_created_date,DATE,YES,None,None,None
8,content_updated_date,DATE,YES,None,None,None
9,content_type,VARCHAR,YES,None,None,None


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*



**Unit of analysis:** one row = one page (content item), on one day — the grain of
`fact_content_daily_performance` is `report_date × client_hash_id × content_hash_id`.

**Time window:** March 2026 (`month=2026-03`) — a mid-panel month, not the sealed final month
(`2026-06`), per the warning that the final month is the natural outcome window for any
past→future label.

Verified below with two queries: the grain claim, and the row count / date span.

In [7]:
grain_check = con.sql(f"""
    SELECT content_hash_id, client_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY content_hash_id, client_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print(f"Rows violating the page-day grain: {len(grain_check)} (should be 0 — confirms one row = one page-day)")
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating the page-day grain: 0 (should be 0 — confirms one row = one page-day)


,content_hash_id,client_hash_id,report_date,c


In [8]:
counts_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_pages
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print("Confirms the time window claim — should span all of March 2026:")
counts_check

Confirms the time window claim — should span all of March 2026:


,row_count,min_date,max_date,n_clients,n_pages
0,9841378,2026-03-01,2026-03-31,55,331437



*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

## 2. Fields: feature / label / context / excluded

**Feature:** CTR (computed as `gsc_clicks / gsc_impressions`), `gsc_avg_position`,
`content_type`, `word_count`, `content_age_days` (computed as `report_date -
content_created_date`) — all knowable at the decision moment.

**Label / proxy:** CTR-gap score — computed CTR vs. tier-average CTR — kept strictly separate
from the feature set.

**Context:** `content_hash_id`, `client_hash_id`, `report_date`, `keyword_hash_id`,
`url_hash_id` — joining/grouping/deduplication only, never learned from.

**Excluded:** `trend_direction`, `trend_pct` (leakage traps — derived from the outcome);
`is_deleted`/`is_published` used only as filters, never features (moderation state, not a
search signal); any product-decision columns like `health_score`/`priority_score` (not shipped
in this data); rows before `ga4_data_start` (zero-filled, not truly zero — filtered via
`ga4_data_available`).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



Grain and counts/date span were already proven in Section 1. The third required query —
availability, filtered with `IS TRUE` — is below, followed by the five-feature frame and the
leakage trap.

In [9]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_impressions IS NOT NULL AND gsc_impressions > 0 THEN 1 ELSE 0 END) AS rows_with_gsc,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4_true
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print(f"Of the rows in {MONTH}, this many have real GA4 data (IS TRUE, not zero-filled):")
availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Of the rows in 2026-03, this many have real GA4 data (IS TRUE, not zero-filled):


,total_rows,rows_with_gsc,rows_with_ga4_true
0,9841378,3611061.0,413966.0


### Five features for my lane (each: knowable at the decision moment because…)

1. **CTR (`gsc_clicks / gsc_impressions`)** — measured directly from that day's actual
   clicks/impressions, not derived from any future window.
2. **`gsc_avg_position`** — the page's actual search ranking that day, an observed fact.
3. **`content_type`** — static content metadata set when the page was created.
4. **`content_age_days`** (computed as `report_date - content_created_date`) — purely a
   function of creation date vs. the report date, no outcome information involved.
5. **`word_count`** — a static property of the page's current content.

In [10]:
features = con.sql(f"""
    SELECT
        f.content_hash_id,
        f.report_date,
        f.gsc_impressions,
        f.gsc_clicks,
        ROUND(f.gsc_clicks * 1.0 / NULLIF(f.gsc_impressions, 0), 4) AS ctr,
        f.gsc_avg_position,
        c.content_type,
        c.word_count,
        DATE_DIFF('day', c.content_created_date, f.report_date) AS content_age_days
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c
      ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_impressions IS NOT NULL AND f.gsc_impressions > 0
      AND c.is_deleted = FALSE AND c.is_published = TRUE
    LIMIT 20
""").df()
print(f"{len(features)} sample rows shown")
features

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

20 sample rows shown


,content_hash_id,report_date,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,content_type,word_count,content_age_days
0,content_b7e512995f79d5a6,2026-03-01,20,0,0.0000,3.350000,keyword article,<NA>,366
1,content_05597932fe4da067,2026-03-01,1,0,0.0000,0.000000,keyword article,<NA>,366
2,content_7a105f548d9c6916,2026-03-01,125,1,0.0080,4.928000,keyword article,2123,366
3,content_905aa32a0230694e,2026-03-01,7,0,0.0000,4.000000,keyword article,<NA>,366
4,content_a3ea9792f793ec72,2026-03-01,11,0,0.0000,2.272727,keyword article,<NA>,366
5,content_36c36abc7650d7af,2026-03-01,239,1,0.0042,7.347280,keyword article,2546,366
6,content_a7da352b73b02668,2026-03-01,191,0,0.0000,7.832461,keyword article,2330,366
7,content_05434271b257bb68,2026-03-01,55,0,0.0000,3.272727,keyword article,<NA>,366
8,content_d056587ff7faca0c,2026-03-01,77,0,0.0000,5.636364,keyword article,2475,366
9,content_bfd1e41c2af250c8,2026-03-01,2,0,0.0000,4.500000,keyword article,<NA>,366


### 4. The trap — add a label-derived column on purpose, then remove it

The warehouse fact table doesn't ship a pre-computed `trend_pct`/`trend_direction` like the
starter CSV does — so instead I'm demonstrating the same lesson with `ctr` itself: since my
label is literally defined as "ctr below the median," feeding `ctr` back in as a feature is the
purest form of leakage — the model would just be handed the answer. Watching the score jump
when I do this, then removing it, is the same lesson from Notebook 02, just constructed
directly rather than relying on a column that turned out not to exist in this table.

In [15]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

demo_raw = con.sql(f"""
    SELECT f.content_hash_id, f.gsc_clicks, f.gsc_impressions,
           c.word_count, DATE_DIFF('day', c.content_created_date, f.report_date) AS content_age_days
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') f
    JOIN read_parquet('{BASE}/dim_content.parquet') c ON f.content_hash_id = c.content_hash_id
    WHERE f.gsc_impressions >= 20
""").df().dropna(subset=["word_count", "content_age_days"])

# Collapse to ONE ROW PER PAGE (average across the month) so no page appears twice
demo = demo_raw.groupby("content_hash_id").agg(
    gsc_clicks=("gsc_clicks", "sum"),
    gsc_impressions=("gsc_impressions", "sum"),
    word_count=("word_count", "first"),
    content_age_days=("content_age_days", "mean"),
).reset_index()

demo["ctr"] = demo["gsc_clicks"] / demo["gsc_impressions"]
demo["label"] = (demo["ctr"] < demo["ctr"].median()).astype(int)

print(f"{len(demo)} unique pages after collapsing daily rows")

honest_features = ["word_count", "content_age_days"]
X = demo[honest_features].fillna(0)
y = demo["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

tree_honest = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
print(f"Honest score on held-out data: {tree_honest.score(X_test, y_test):.3f}")

demo_leaky = demo[honest_features + ["ctr"]].fillna(0)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(demo_leaky, y, test_size=0.3, random_state=42, stratify=y)

tree_leaky = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train_l, y_train_l)
print(f"Leaky score on held-out data (with ctr sneaked in): {tree_leaky.score(X_test_l, y_test_l):.3f}  <- jumps toward perfect")

print("\nDeleting the leaked column and keeping only the honest number.")
del demo_leaky, X_train_l, X_test_l

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

66147 unique pages after collapsing daily rows
Honest score on held-out data: 0.580
Leaky score on held-out data (with ctr sneaked in): 1.000  <- jumps toward perfect

Deleting the leaked column and keeping only the honest number.


**What happened:** the honest model (word count + content age only) scored 0.580 on held-out
pages — barely above chance, which makes sense since neither feature is a strong CTR driver on
its own. The moment I added `ctr` itself as a feature — the exact value my label is
thresholded from — the score jumped to 1.000. That's not a better model; it's the model being
handed the answer.

**A bug I found and fixed along the way:** my first two attempts at this also scored 1.000 on
the "honest" side, which was wrong. The cause was leakage of a different kind — I was splitting
train/test by row (page-day), not by page, so the same page's near-identical features showed up
in both train and test on different days. The model wasn't learning anything, it was
memorizing pages it had already seen. Fixing this required collapsing to one row per page before
splitting. This is a real, general lesson about this data: daily rows for the same page are
highly correlated, so any row-level split risks silent leakage — splits should be done by page
(or by client, for cross-client generalization) rather than by row.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*



**Named limitation:** client history is an unbalanced panel — only 9 of 70 clients have 12+
months of history, and `gsc_data_start`/`ga4_data_start` differ per client. For March 2026,
some clients may not have started tracking yet — absence isn't "zero activity," it's
"not yet measured." This slice can't be treated as representative without checking
`dim_clients` start dates first.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

##ALL DONE